# Notebook 17 — Neutral-expression target prompt pilot (FlowEdit SD3)

**Status: NOT EXECUTED. Pilot design, not yet final validation.** Tests a neutral-*expression* target instead of the N07 **same-expression / different-person** target. Neutral expression and preservation of the original emotion are competing objectives, so utility is expected to change. Do **not** call this an identity-only intervention or infer privacy performance from generated images alone.

Frozen references: N07 originals; N07 guidance 5.5 seed 42/123; T=40, source guidance=3.5, target guidance=5.5, n_min=0,n_max=33,n_avg=1; original FlowEdit commit `85594626161caca9dafa68592c1ffa21db87a806` and SD3 revision `ea42f8cef0f178587cf766dc8129abd379c90671`. The repo checkout is pinned; *model revision pinning must be confirmed at runtime in the runner* before labeling the generation fully comparable. No scoring is claimed by this notebook.

**Pilot:** 12 held-out sources (two for each of six emotions), *excluding* all 12 NB06c tuning sources, both original seeds; 24 edits. If the preflight or runner compatibility fails, **fail loudly**, and leave a status report rather than inventing outputs. Only after pilot review extend to the other 96 held-out sources.

Raw images and per-source rows remain in private Drive, never public GitHub. Need a later evaluator notebook to calculate FER, V3 privacy, and image-quality metrics with exactly the same fixed evaluators.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, re, sys, subprocess, time, hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, yaml
RUNS=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=RUNS/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
assert N07.is_dir() and (N07/'selected_sources.csv').is_file()
assert (N07/'sources').is_dir() and (N07/'flowedit_configs').is_dir()
tuning={'source_000','source_001','source_005','source_006','source_010','source_011','source_015','source_016','source_020','source_021','source_025','source_026'}
sources=pd.read_csv(N07/'selected_sources.csv')
assert sources.source_id.is_unique and len(sources)==120 and sources.emotion.nunique()==6
eligible=sources[~sources.source_id.isin(tuning)].copy()
assert len(eligible)==108 and (eligible.groupby('emotion').size()>=2).all()
# deterministic source order, predeclared without reading results
pilot=pd.concat([v.sort_values('source_id').head(2) for _,v in eligible.groupby('emotion',sort=True)],ignore_index=True)
assert len(pilot)==12 and pilot.source_id.nunique()==12
assert not set(pilot.source_id)&tuning
print(pilot[['source_id','emotion']].to_string(index=False))
ts=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=RUNS/('nb17_neutral_target_pilot_'+ts)
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'configs').mkdir();(OUT/'images').mkdir();(OUT/'logs').mkdir()
pilot[['source_id','emotion']].to_csv(OUT/'private_pilot_sources.csv',index=False)
print('Private output:',OUT)


In [ ]:
# Match N07 source prompts and freeze a single target-prompt policy.
# Read existing (source-specific) N07 YAML, instead of guessing the original prompting.
src_cfg=[]
for row in pilot.itertuples():
    sid=row.source_id
    f=N07/'flowedit_configs'/f'nb07_{sid}_flowedit_g5p5_s42_f1142664_edits.yaml'
    assert f.is_file(), f'Missing N07 evidence: {f}'
    z=yaml.safe_load(f.read_text()); assert len(z)==1
    old=z[0]; orig=old['source_prompt']; previous=old['target_prompts'][0]
    assert str(row.emotion) in orig and 'same' in previous.lower() and 'different person' in previous.lower()
    neutral=('A realistic close-up portrait photograph of a different person with a neutral, '
             'relaxed facial expression, keeping similar head angle and eye gaze to the original. Photorealistic.')
    src_cfg.append({'source_id':sid,'emotion':row.emotion,'source_prompt':orig,
                    'n07_target_prompt':previous,'neutral_target_prompt':neutral})
# Freeze before executing model. No face images or prompts copied into public repo.
(OUT/'private_prompt_manifest.json').write_text(json.dumps(src_cfg,indent=2))
print('Frozen prompts:',len(src_cfg),'(source prompt remains N07; only target prompt differs)')


In [ ]:
# Install FlowEdit *pinned* code. GPU runtime and HF SD3 access are required.
import torch
assert torch.cuda.is_available(), 'GPU required: Runtime > Change runtime type > T4 GPU'
import importlib.metadata
# Keep tested FlowEdit dependency family. Restart runtime if dependency installations request it.
subprocess.run([sys.executable,'-m','pip','-q','install','diffusers==0.30.1','transformers==4.44.2',
                'accelerate','sentencepiece','protobuf','safetensors','huggingface_hub'],check=True)
work=Path('/content/flowedit_nb17')
if not work.exists():
    subprocess.run(['git','clone','--quiet','https://github.com/fallenshock/FlowEdit.git',str(work)],check=True)
subprocess.run(['git','-C',str(work),'fetch','--quiet','origin'],check=True)
subprocess.run(['git','-C',str(work),'checkout','--quiet','85594626161caca9dafa68592c1ffa21db87a806'],check=True)
revision=subprocess.check_output(['git','-C',str(work),'rev-parse','HEAD'],text=True).strip()
assert revision=='85594626161caca9dafa68592c1ffa21db87a806'
assert (work/'run_script.py').is_file()
print('FlowEdit code revision:',revision)
print('NOTICE: Stable Diffusion 3 model revision is not yet independently pinned/verified in run_script; status is conditional.')


In [ ]:
# Generate all 24 edits with independent YAML files and run receipts.
# The official FlowEdit runner consumes images under its example_images directory.
from PIL import Image
import shutil
(work/'example_images').mkdir(exist_ok=True)
completed=[]
for cfg in src_cfg:
    sid=cfg['source_id']
    source_file=N07/'sources'/f'{sid}.png'
    assert source_file.is_file(),f'Frozen N07 image missing: {source_file}'
    prepared_hash=hashlib.sha256(source_file.read_bytes()).hexdigest()
    expected=sources.set_index('source_id').loc[sid,'prepared_source_sha256']
    assert prepared_hash==expected,f'Source SHA mismatch for {sid}'
    for seed in (42,123):
        run=f'nb17_{sid}_neutral_g5p5_s{seed}'
        img_name=f'{run}.png'
        shutil.copy2(source_file,work/'example_images'/img_name)
        edit=[{'input_img':f'example_images/{img_name}',
               'source_prompt':cfg['source_prompt'],
               'target_prompts':[cfg['neutral_target_prompt']],
               'target_codes':['neutral_expression_different_person']}]
        experiment=[{'exp_name':run,'dataset_yaml':str(OUT/'configs'/f'{run}_edits.yaml'),
                     'model_type':'SD3','sampler_type':'FlowEditSD3','T_steps':40,
                     'src_guidance_scale':3.5,'tar_guidance_scale':5.5,
                     'n_min':0,'n_max':33,'n_avg':1,'seed':seed}]
        edit_path=OUT/'configs'/f'{run}_edits.yaml'; exp_path=OUT/'configs'/f'{run}_exp.yaml'
        edit_path.write_text(yaml.safe_dump(edit,sort_keys=False));exp_path.write_text(yaml.safe_dump(experiment,sort_keys=False))
        command=[sys.executable,'run_script.py','--exp_yaml',str(exp_path)]
        log_path=OUT/'logs'/f'{run}.txt'
        t=time.time()
        with log_path.open('w') as log:
            p=subprocess.run(command,cwd=str(work),stdout=log,stderr=subprocess.STDOUT)
        rec={'source_id':sid,'emotion':cfg['emotion'],'seed':seed,'condition':'neutral_expression_target_g5p5',
             'run_name':run,'prepared_source_sha256':prepared_hash,'exit_code':p.returncode,
             'runtime_seconds':round(time.time()-t,2),'runner_commit':revision,
             'sd3_revision_verified':False,'log':str(log_path)}
        completed.append(rec)
        pd.DataFrame(completed).to_csv(OUT/'private_execution_receipts.csv',index=False)
        if p.returncode!=0:
            raise RuntimeError(f'{run} failed; inspect {log_path}. Existing receipts persisted.')
assert len(completed)==24
print('Runner jobs finished:',len(completed),'— locate and verify actual generated files before marking generation complete.')


In [ ]:
# Record outputs without assuming FlowEdit's output directory or filename conventions.
# Fail until every generated output is physically identified, checked, and privately copied.
from PIL import Image
import shutil
all_png=list(work.rglob('*.png'))
records=[]
for rec in completed:
    # Ignore the copied input. Match runner's target code and unique exp_name.
    candidates=[p for p in all_png if rec['run_name'] in str(p) and 'neutral_expression_different_person' in p.name
                and '/example_images/' not in str(p)]
    if len(candidates)!=1:
        rec['output_resolution']='NEEDS_MANUAL_RUNNER_FILENAME_INSPECTION'
        rec['candidate_paths']=';'.join(map(str,candidates))
        records.append(rec)
        continue
    image=Image.open(candidates[0]).convert('RGB')
    assert image.width>=256 and image.height>=256
    final=OUT/'images'/f"{rec['run_name']}.png"
    shutil.copy2(candidates[0],final)
    rec['output_resolution']='VERIFIED';rec['output_sha256']=hashlib.sha256(final.read_bytes()).hexdigest()
    rec['output_path']=str(final)
    records.append(rec)
pd.DataFrame(records).to_csv(OUT/'private_generation_results.csv',index=False)
status={'jobs':len(records),'verified_images':sum(x['output_resolution']=='VERIFIED' for x in records),
        'model_revision_verification':'PENDING_RUNNER_INSPECTION',
        'privacy_and_emotion_evaluation':'NOT_RUN',
        'experiment_type':'NEUTRAL_EXPRESSION_PROMPT_ABLATION_NOT_ANONYMIZATION_PROOF'}
(OUT/'status.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))
assert status['verified_images']==24,'Review generated-image filename mapping in private_generation_results.csv before proceeding.'
